# Exp 012b: Perch Temporal Ablation

Ablation follow-up to `exp_012`. The first light temporal branch looked promising by fold means but failed on honest pooled OOF. This notebook strips the downstream stack back to a small ladder of simpler models so we can isolate **which component is actually hurting**.


## Hypothesis

The negative `exp_012` result may come from over-complexity rather than from the Perch direction itself. We therefore compare a compact ablation ladder on the same trusted `59` fully labeled files:

1. `raw_perch`
2. `pooled_mlp_rawfeat`: no sequence model, no prototype head, no gated fusion
3. `ssm_linear`: sequence model, but still no prototype head and no gated fusion
4. `ssm_linear_rawfeat`: sequence model plus raw Perch score features, still without gated fusion

Primary decision metric: **grouped pooled OOF macro ROC-AUC**.


In [1]:
from __future__ import annotations

import json
import os
import random
import re
from dataclasses import asdict, dataclass
from pathlib import Path
import typing as tp

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold


In [2]:
ROOT = Path('/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026')
DATA = ROOT / 'data' / 'birdclef-2026'
PERCH_META_DIR = ROOT / 'data' / 'perch_meta'
EXPERIMENTS = ROOT / 'experiments'
OUTPUT_DIR = EXPERIMENTS / 'outputs' / 'exp_012b_perch_temporal_ablation'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


@dataclass
class Config:
    experiment_id: str = 'exp_012b'
    experiment_name: str = 'perch_temporal_ablation'
    random_seed: int = 1088
    n_splits: int = 3

    d_model: int = 192
    raw_dim: int = 64
    meta_dim: int = 16
    d_state: int = 16
    n_ssm_layers: int = 2
    dropout: float = 0.15

    n_epochs: int = 35
    lr: float = 7.5e-4
    weight_decay: float = 2e-3
    patience: int = 8
    pos_weight_cap: float = 30.0

    save_oof_outputs: bool = True


CFG = Config()
RUN_OOF = True
RUN_FINAL_TRAIN = False
BEST_VARIANT_FOR_FINAL = 'ssm_linear_rawfeat'


In [3]:
def set_random_seed(seed: int) -> None:
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def pick_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device('cuda')
    if getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')


def save_json(payload: dict[str, tp.Any], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=False))


set_random_seed(CFG.random_seed)
device = pick_device()
print({
    'root': str(ROOT),
    'perch_meta_dir_exists': PERCH_META_DIR.exists(),
    'device': str(device),
    'output_dir': str(OUTPUT_DIR),
})


{'root': '/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026', 'perch_meta_dir_exists': True, 'device': 'mps', 'output_dir': '/Users/yaroslav/Documents/Kaggle/BirdCLEF_2026/experiments/outputs/exp_012b_perch_temporal_ablation'}


In [4]:
taxonomy = pd.read_csv(DATA / 'taxonomy.csv')
sample_sub = pd.read_csv(DATA / 'sample_submission.csv')
soundscape_labels = pd.read_csv(DATA / 'train_soundscapes_labels.csv').drop_duplicates().reset_index(drop=True)

PRIMARY_LABELS = sample_sub.columns[1:].tolist()
N_CLASSES = len(PRIMARY_LABELS)
label_to_idx = {label: idx for idx, label in enumerate(PRIMARY_LABELS)}


def parse_label_tokens(x: str) -> list[str]:
    if pd.isna(x):
        return []
    return [token.strip() for token in str(x).split(';') if token.strip()]


def union_labels(series: pd.Series) -> list[str]:
    labels = sorted(set(lbl for value in series for lbl in parse_label_tokens(value) if lbl in label_to_idx))
    return labels


FNAME_RE = re.compile(r'BC2026_(?:Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg')

def parse_soundscape_filename(name: str) -> dict[str, tp.Any]:
    match = FNAME_RE.match(name)
    if not match:
        return {'site': None, 'hour_utc': -1, 'file_id': None}
    file_id, site, ymd, hms = match.groups()
    return {'site': site, 'hour_utc': int(hms[:2]), 'file_id': file_id}


sc_clean = (
    soundscape_labels
    .groupby(['filename', 'start', 'end'])['primary_label']
    .apply(union_labels)
    .reset_index(name='label_list')
)
sc_clean['start_sec'] = pd.to_timedelta(sc_clean['start']).dt.total_seconds().astype(int)
sc_clean['end_sec'] = pd.to_timedelta(sc_clean['end']).dt.total_seconds().astype(int)
sc_clean['row_id'] = sc_clean['filename'].str.replace('.ogg', '', regex=False) + '_' + sc_clean['end_sec'].astype(str)
meta = sc_clean['filename'].apply(parse_soundscape_filename).apply(pd.Series)
sc_clean = pd.concat([sc_clean, meta], axis=1)

windows_per_file = sc_clean.groupby('filename').size()
full_files = sorted(windows_per_file[windows_per_file == 12].index.tolist())
sc_clean['file_fully_labeled'] = sc_clean['filename'].isin(full_files)

Y_SC = np.zeros((len(sc_clean), N_CLASSES), dtype=np.uint8)
for row_idx, labels in enumerate(sc_clean['label_list']):
    for label in labels:
        if label in label_to_idx:
            Y_SC[row_idx, label_to_idx[label]] = 1

full_truth = (
    sc_clean[sc_clean['file_fully_labeled']]
    .sort_values(['filename', 'end_sec'])
    .reset_index(drop=False)
)

print({
    'full_files': len(full_files),
    'trusted_windows': len(full_truth),
    'active_classes_in_trusted_set': int((Y_SC[full_truth['index'].to_numpy()].sum(axis=0) > 0).sum()),
})


{'full_files': 59, 'trusted_windows': 708, 'active_classes_in_trusted_set': 71}


In [5]:
meta_full = pd.read_parquet(PERCH_META_DIR / 'full_perch_meta.parquet')
arr = np.load(PERCH_META_DIR / 'full_perch_arrays.npz')
scores_full_raw = arr['scores_full_raw'].astype(np.float32)
emb_full = arr['emb_full'].astype(np.float32)

full_truth_aligned = full_truth.set_index('row_id').loc[meta_full['row_id']].reset_index()
Y_FULL = Y_SC[full_truth_aligned['index'].to_numpy()].astype(np.float32)

assert np.all(full_truth_aligned['filename'].values == meta_full['filename'].values)
assert np.all(full_truth_aligned['row_id'].values == meta_full['row_id'].values)

print({
    'meta_full_shape': tuple(meta_full.shape),
    'scores_full_raw_shape': tuple(scores_full_raw.shape),
    'emb_full_shape': tuple(emb_full.shape),
    'y_full_shape': tuple(Y_FULL.shape),
})


{'meta_full_shape': (708, 4), 'scores_full_raw_shape': (708, 234), 'emb_full_shape': (708, 1536), 'y_full_shape': (708, 234)}


In [6]:
def macro_auc_skip_empty(y_true: np.ndarray, y_score: np.ndarray) -> float:
    positive_mask = y_true.sum(axis=0) > 0
    negative_mask = (y_true.shape[0] - y_true.sum(axis=0)) > 0
    scored_mask = positive_mask & negative_mask
    if not scored_mask.any():
        return float('nan')
    return float(roc_auc_score(y_true[:, scored_mask], y_score[:, scored_mask], average='macro'))


def reshape_to_files(flat_array: np.ndarray, meta_df: pd.DataFrame, n_windows: int = 12):
    filenames = meta_df['filename'].to_numpy()
    unique_files = []
    seen = set()
    for filename in filenames:
        if filename not in seen:
            unique_files.append(filename)
            seen.add(filename)
    n_files = len(unique_files)
    assert len(flat_array) == n_files * n_windows, (len(flat_array), n_files, n_windows)
    new_shape = (n_files, n_windows) + flat_array.shape[1:]
    return flat_array.reshape(new_shape), unique_files


def build_site_mapping(meta_df: pd.DataFrame):
    sites = sorted(meta_df['site'].dropna().astype(str).unique().tolist())
    site_to_idx = {site: idx + 1 for idx, site in enumerate(sites)}
    return site_to_idx, len(sites) + 1


def get_file_metadata(meta_df: pd.DataFrame, file_list: list[str], site_to_idx: dict[str, int], n_sites_max: int):
    file_to_row = {}
    filenames = meta_df['filename'].to_numpy()
    sites = meta_df['site'].astype(str).to_numpy()
    hours = meta_df['hour_utc'].astype(int).to_numpy()
    for idx, filename in enumerate(filenames):
        if filename not in file_to_row:
            file_to_row[filename] = idx

    site_ids = np.zeros(len(file_list), dtype=np.int64)
    hour_ids = np.zeros(len(file_list), dtype=np.int64)
    group_ids = np.empty(len(file_list), dtype=object)
    for file_idx, filename in enumerate(file_list):
        row = file_to_row[filename]
        site = sites[row]
        hour = hours[row]
        site_ids[file_idx] = min(site_to_idx.get(site, 0), n_sites_max - 1)
        hour_ids[file_idx] = hour % 24
        group_ids[file_idx] = site
    return site_ids, hour_ids, group_ids


emb_files, file_list = reshape_to_files(emb_full, meta_full)
raw_files, _ = reshape_to_files(scores_full_raw, meta_full)
labels_files, _ = reshape_to_files(Y_FULL, meta_full)
site_to_idx, n_sites = build_site_mapping(meta_full)
site_ids_all, hours_all, group_ids_all = get_file_metadata(meta_full, file_list, site_to_idx, n_sites)

raw_perch_auc = macro_auc_skip_empty(Y_FULL, scores_full_raw)
legacy_probe_snapshot = ROOT / 'experiments' / 'outputs' / 'exp_003_perch_downstream_reproduction' / 'result_snapshot.json'
legacy_probe_auc = None
if legacy_probe_snapshot.exists():
    legacy_probe_auc = json.loads(legacy_probe_snapshot.read_text()).get('probe_oof_auc')

print({
    'emb_files_shape': tuple(emb_files.shape),
    'raw_files_shape': tuple(raw_files.shape),
    'labels_files_shape': tuple(labels_files.shape),
    'n_sites': int(n_sites),
    'raw_perch_auc': raw_perch_auc,
    'legacy_exp003_probe_auc': legacy_probe_auc,
})


{'emb_files_shape': (59, 12, 1536), 'raw_files_shape': (59, 12, 234), 'labels_files_shape': (59, 12, 234), 'n_sites': 9, 'raw_perch_auc': 0.7390178442294307, 'legacy_exp003_probe_auc': 0.8353024140080767}


In [7]:
class SelectiveSSM(nn.Module):
    def __init__(self, d_model: int, d_state: int = 16, d_conv: int = 4):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.in_proj = nn.Linear(d_model, 2 * d_model, bias=False)
        self.conv1d = nn.Conv1d(d_model, d_model, d_conv, padding=d_conv - 1, groups=d_model)
        self.dt_proj = nn.Linear(d_model, d_model, bias=True)
        A = torch.arange(1, d_state + 1, dtype=torch.float32).unsqueeze(0).expand(d_model, -1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(d_model))
        self.B_proj = nn.Linear(d_model, d_state, bias=False)
        self.C_proj = nn.Linear(d_model, d_state, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, steps, d_model = x.shape
        xz = self.in_proj(x)
        x_ssm, _ = xz.chunk(2, dim=-1)
        x_conv = self.conv1d(x_ssm.transpose(1, 2))[:, :, :steps].transpose(1, 2)
        x_conv = F.silu(x_conv)
        dt = F.softplus(self.dt_proj(x_conv))
        A = -torch.exp(self.A_log)
        B = self.B_proj(x_conv)
        C = self.C_proj(x_conv)
        h = torch.zeros(batch_size, d_model, self.d_state, device=x.device)
        outputs = []
        for step in range(steps):
            dt_t = dt[:, step, :]
            dA = torch.exp(A[None, :, :] * dt_t[:, :, None])
            dB = dt_t[:, :, None] * B[:, step, None, :]
            h = h * dA + x[:, step, :, None] * dB
            y_t = (h * C[:, step, None, :]).sum(-1)
            outputs.append(y_t)
        y = torch.stack(outputs, dim=1)
        return y + x * self.D[None, None, :]


class PooledMLPRawFeat(nn.Module):
    def __init__(self, d_input: int = 1536, n_classes: int = N_CLASSES):
        super().__init__()
        self.win_proj = nn.Sequential(
            nn.Linear(d_input, CFG.d_model),
            nn.LayerNorm(CFG.d_model),
            nn.GELU(),
            nn.Dropout(CFG.dropout),
        )
        self.ctx_proj = nn.Sequential(
            nn.Linear(d_input, CFG.d_model),
            nn.LayerNorm(CFG.d_model),
            nn.GELU(),
            nn.Dropout(CFG.dropout),
        )
        self.raw_proj = nn.Sequential(
            nn.Linear(n_classes, CFG.raw_dim),
            nn.LayerNorm(CFG.raw_dim),
            nn.GELU(),
            nn.Dropout(CFG.dropout),
        )
        self.site_emb = nn.Embedding(n_sites, CFG.meta_dim)
        self.hour_emb = nn.Embedding(24, CFG.meta_dim)
        self.head = nn.Sequential(
            nn.Linear(2 * CFG.d_model + CFG.raw_dim + 2 * CFG.meta_dim, CFG.d_model),
            nn.LayerNorm(CFG.d_model),
            nn.GELU(),
            nn.Dropout(CFG.dropout),
            nn.Linear(CFG.d_model, n_classes),
        )

    def forward(self, emb: torch.Tensor, raw_scores: torch.Tensor, site_ids: torch.Tensor, hours: torch.Tensor) -> torch.Tensor:
        steps = emb.shape[1]
        win_h = self.win_proj(emb)
        ctx_h = self.ctx_proj(emb.mean(dim=1))[:, None, :].expand(-1, steps, -1)
        raw_h = self.raw_proj(raw_scores)
        meta = torch.cat([self.site_emb(site_ids), self.hour_emb(hours)], dim=-1)[:, None, :].expand(-1, steps, -1)
        feat = torch.cat([win_h, ctx_h, raw_h, meta], dim=-1)
        return self.head(feat)


class SSMLinearModel(nn.Module):
    def __init__(self, d_input: int = 1536, n_classes: int = N_CLASSES, n_windows: int = 12, include_raw_features: bool = False):
        super().__init__()
        self.include_raw_features = include_raw_features
        self.input_proj = nn.Sequential(
            nn.Linear(d_input, CFG.d_model),
            nn.LayerNorm(CFG.d_model),
            nn.GELU(),
            nn.Dropout(CFG.dropout),
        )
        self.pos_enc = nn.Parameter(torch.randn(1, n_windows, CFG.d_model) * 0.02)
        self.site_emb = nn.Embedding(n_sites, CFG.meta_dim)
        self.hour_emb = nn.Embedding(24, CFG.meta_dim)
        self.meta_proj = nn.Linear(2 * CFG.meta_dim, CFG.d_model)

        self.ssm_fwd = nn.ModuleList([SelectiveSSM(CFG.d_model, CFG.d_state) for _ in range(CFG.n_ssm_layers)])
        self.ssm_bwd = nn.ModuleList([SelectiveSSM(CFG.d_model, CFG.d_state) for _ in range(CFG.n_ssm_layers)])
        self.ssm_merge = nn.ModuleList([nn.Linear(2 * CFG.d_model, CFG.d_model) for _ in range(CFG.n_ssm_layers)])
        self.ssm_norm = nn.ModuleList([nn.LayerNorm(CFG.d_model) for _ in range(CFG.n_ssm_layers)])
        self.ssm_drop = nn.Dropout(CFG.dropout)

        if self.include_raw_features:
            self.raw_proj = nn.Sequential(
                nn.Linear(n_classes, CFG.raw_dim),
                nn.LayerNorm(CFG.raw_dim),
                nn.GELU(),
                nn.Dropout(CFG.dropout),
            )
            head_in = CFG.d_model + CFG.raw_dim
        else:
            head_in = CFG.d_model

        self.head = nn.Sequential(
            nn.Linear(head_in, CFG.d_model),
            nn.LayerNorm(CFG.d_model),
            nn.GELU(),
            nn.Dropout(CFG.dropout),
            nn.Linear(CFG.d_model, n_classes),
        )

    def forward(self, emb: torch.Tensor, raw_scores: torch.Tensor, site_ids: torch.Tensor, hours: torch.Tensor) -> torch.Tensor:
        h = self.input_proj(emb)
        h = h + self.pos_enc[:, :h.shape[1], :]
        meta = self.meta_proj(torch.cat([self.site_emb(site_ids), self.hour_emb(hours)], dim=-1))
        h = h + meta[:, None, :]

        for fwd, bwd, merge, norm in zip(self.ssm_fwd, self.ssm_bwd, self.ssm_merge, self.ssm_norm):
            residual = h
            h_f = fwd(h)
            h_b = bwd(h.flip(1)).flip(1)
            h = merge(torch.cat([h_f, h_b], dim=-1))
            h = self.ssm_drop(h)
            h = norm(h + residual)

        if self.include_raw_features:
            raw_h = self.raw_proj(raw_scores)
            h = torch.cat([h, raw_h], dim=-1)
        return self.head(h)


VARIANT_ORDER = ('raw_perch', 'pooled_mlp_rawfeat', 'ssm_linear', 'ssm_linear_rawfeat')


def build_model(variant: str) -> nn.Module:
    if variant == 'pooled_mlp_rawfeat':
        return PooledMLPRawFeat()
    if variant == 'ssm_linear':
        return SSMLinearModel(include_raw_features=False)
    if variant == 'ssm_linear_rawfeat':
        return SSMLinearModel(include_raw_features=True)
    raise ValueError(f'Unknown variant: {variant}')


In [8]:
def make_pos_weight(labels_tr: torch.Tensor) -> torch.Tensor:
    pos_counts = labels_tr.sum(dim=(0, 1))
    total = labels_tr.shape[0] * labels_tr.shape[1]
    pos_weight = (total - pos_counts) / (pos_counts + 1.0)
    return pos_weight.clamp(max=CFG.pos_weight_cap)


def train_variant(
    variant: str,
    emb_train: np.ndarray,
    raw_train: np.ndarray,
    labels_train: np.ndarray,
    site_train: np.ndarray,
    hour_train: np.ndarray,
    emb_valid: np.ndarray,
    raw_valid: np.ndarray,
    labels_valid: np.ndarray,
    site_valid: np.ndarray,
    hour_valid: np.ndarray,
) -> tuple[nn.Module, dict[str, list[float]]]:
    model = build_model(variant).to(device)

    emb_tr = torch.tensor(emb_train, dtype=torch.float32, device=device)
    raw_tr = torch.tensor(raw_train, dtype=torch.float32, device=device)
    labels_tr = torch.tensor(labels_train, dtype=torch.float32, device=device)
    site_tr = torch.tensor(site_train, dtype=torch.long, device=device)
    hour_tr = torch.tensor(hour_train, dtype=torch.long, device=device)

    emb_va = torch.tensor(emb_valid, dtype=torch.float32, device=device)
    raw_va = torch.tensor(raw_valid, dtype=torch.float32, device=device)
    labels_va = torch.tensor(labels_valid, dtype=torch.float32, device=device)
    site_va = torch.tensor(site_valid, dtype=torch.long, device=device)
    hour_va = torch.tensor(hour_valid, dtype=torch.long, device=device)

    pos_weight = make_pos_weight(labels_tr)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.n_epochs)

    best_state = None
    best_valid_auc = -float('inf')
    wait = 0
    history = {'train_loss': [], 'valid_loss': [], 'valid_auc': []}

    for epoch in range(1, CFG.n_epochs + 1):
        model.train()
        train_logits = model(emb_tr, raw_tr, site_tr, hour_tr)
        train_loss = F.binary_cross_entropy_with_logits(train_logits, labels_tr, pos_weight=pos_weight[None, None, :])

        optimizer.zero_grad(set_to_none=True)
        train_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            valid_logits = model(emb_va, raw_va, site_va, hour_va)
            valid_loss = F.binary_cross_entropy_with_logits(valid_logits, labels_va, pos_weight=pos_weight[None, None, :])
            valid_probs = torch.sigmoid(valid_logits).detach().cpu().numpy().reshape(-1, N_CLASSES)

        valid_true = labels_valid.reshape(-1, N_CLASSES)
        valid_auc = macro_auc_skip_empty(valid_true, valid_probs)

        history['train_loss'].append(float(train_loss.item()))
        history['valid_loss'].append(float(valid_loss.item()))
        history['valid_auc'].append(float(valid_auc))

        if valid_auc > best_valid_auc:
            best_valid_auc = float(valid_auc)
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            wait = 0
        else:
            wait += 1

        print({
            'variant': variant,
            'epoch': epoch,
            'train_loss': float(train_loss.item()),
            'valid_loss': float(valid_loss.item()),
            'valid_auc': float(valid_auc),
        })
        if wait >= CFG.patience:
            print(f'Early stop for {variant} at epoch {epoch}')
            break

    assert best_state is not None
    model.load_state_dict(best_state)
    return model, history


def run_grouped_oof() -> dict[str, tp.Any]:
    splitter = GroupKFold(n_splits=CFG.n_splits)
    oof_probs = {
        'raw_perch': raw_files.astype(np.float32, copy=True),
        'pooled_mlp_rawfeat': np.zeros_like(labels_files, dtype=np.float32),
        'ssm_linear': np.zeros_like(labels_files, dtype=np.float32),
        'ssm_linear_rawfeat': np.zeros_like(labels_files, dtype=np.float32),
    }
    fold_rows = []

    for fold, (train_idx, valid_idx) in enumerate(splitter.split(emb_files, groups=group_ids_all), 1):
        print(f'\n=== Fold {fold} ===')
        valid_true = labels_files[valid_idx].reshape(-1, N_CLASSES)
        raw_fold_auc = macro_auc_skip_empty(valid_true, raw_files[valid_idx].reshape(-1, N_CLASSES))
        fold_rows.append({
            'fold': fold,
            'variant': 'raw_perch',
            'valid_files': int(len(valid_idx)),
            'valid_sites': sorted(set(group_ids_all[valid_idx].tolist())),
            'best_valid_auc': float(raw_fold_auc),
            'oof_auc': float(raw_fold_auc),
            'epochs_ran': 0,
        })

        for variant in VARIANT_ORDER[1:]:
            model, history = train_variant(
                variant=variant,
                emb_train=emb_files[train_idx],
                raw_train=raw_files[train_idx],
                labels_train=labels_files[train_idx],
                site_train=site_ids_all[train_idx],
                hour_train=hours_all[train_idx],
                emb_valid=emb_files[valid_idx],
                raw_valid=raw_files[valid_idx],
                labels_valid=labels_files[valid_idx],
                site_valid=site_ids_all[valid_idx],
                hour_valid=hours_all[valid_idx],
            )
            model.eval()
            with torch.no_grad():
                emb_va = torch.tensor(emb_files[valid_idx], dtype=torch.float32, device=device)
                raw_va = torch.tensor(raw_files[valid_idx], dtype=torch.float32, device=device)
                site_va = torch.tensor(site_ids_all[valid_idx], dtype=torch.long, device=device)
                hour_va = torch.tensor(hours_all[valid_idx], dtype=torch.long, device=device)
                probs = torch.sigmoid(model(emb_va, raw_va, site_va, hour_va)).detach().cpu().numpy().astype(np.float32)
            oof_probs[variant][valid_idx] = probs
            fold_auc = macro_auc_skip_empty(valid_true, probs.reshape(-1, N_CLASSES))
            fold_rows.append({
                'fold': fold,
                'variant': variant,
                'valid_files': int(len(valid_idx)),
                'valid_sites': sorted(set(group_ids_all[valid_idx].tolist())),
                'best_valid_auc': float(max(history['valid_auc'])),
                'oof_auc': float(fold_auc),
                'epochs_ran': int(len(history['valid_auc'])),
            })

    pooled_auc = {
        variant: macro_auc_skip_empty(labels_files.reshape(-1, N_CLASSES), probs.reshape(-1, N_CLASSES))
        for variant, probs in oof_probs.items()
    }
    best_variant = max(pooled_auc, key=pooled_auc.get)
    return {
        'oof_probs': oof_probs,
        'pooled_auc': pooled_auc,
        'best_variant': best_variant,
        'fold_summary': fold_rows,
    }


if RUN_OOF:
    oof_result = run_grouped_oof()
    pooled_auc = oof_result['pooled_auc']
    best_variant = oof_result['best_variant']
    result_snapshot = {
        'experiment_id': CFG.experiment_id,
        'experiment_name': CFG.experiment_name,
        'n_splits': CFG.n_splits,
        'raw_perch_auc': float(pooled_auc['raw_perch']),
        'legacy_exp003_probe_auc': legacy_probe_auc,
        'variant_auc': {key: float(value) for key, value in pooled_auc.items()},
        'best_variant': best_variant,
        'best_variant_auc': float(pooled_auc[best_variant]),
        'delta_best_vs_raw': float(pooled_auc[best_variant] - pooled_auc['raw_perch']),
        'delta_best_vs_exp003': None if legacy_probe_auc is None else float(pooled_auc[best_variant] - legacy_probe_auc),
        'n_files': int(len(file_list)),
        'n_sites': int(len(set(group_ids_all.tolist()))),
        'model': {
            'd_model': CFG.d_model,
            'raw_dim': CFG.raw_dim,
            'meta_dim': CFG.meta_dim,
            'd_state': CFG.d_state,
            'n_ssm_layers': CFG.n_ssm_layers,
            'dropout': CFG.dropout,
        },
        'output_dir': str(OUTPUT_DIR),
    }
    save_json(result_snapshot, OUTPUT_DIR / 'result_snapshot.json')
    pd.DataFrame(oof_result['fold_summary']).to_csv(OUTPUT_DIR / 'fold_summary.csv', index=False)
    pd.DataFrame([
        {'variant': variant, 'pooled_oof_auc': float(score)}
        for variant, score in pooled_auc.items()
    ]).sort_values('pooled_oof_auc', ascending=False).to_csv(OUTPUT_DIR / 'variant_auc.csv', index=False)
    if CFG.save_oof_outputs:
        np.savez_compressed(
            OUTPUT_DIR / 'oof_outputs.npz',
            labels=labels_files.astype(np.float32),
            raw_perch=oof_result['oof_probs']['raw_perch'].astype(np.float32),
            pooled_mlp_rawfeat=oof_result['oof_probs']['pooled_mlp_rawfeat'].astype(np.float32),
            ssm_linear=oof_result['oof_probs']['ssm_linear'].astype(np.float32),
            ssm_linear_rawfeat=oof_result['oof_probs']['ssm_linear_rawfeat'].astype(np.float32),
        )
        pd.DataFrame({'filename': file_list, 'site': group_ids_all, 'hour_utc': hours_all}).to_csv(OUTPUT_DIR / 'file_meta.csv', index=False)
    print(json.dumps(result_snapshot, indent=2))
else:
    print('OOF run skipped. Set RUN_OOF = True to compare the ablation variants.')



=== Fold 1 ===
{'variant': 'pooled_mlp_rawfeat', 'epoch': 1, 'train_loss': 0.84348064661026, 'valid_loss': 0.9344828724861145, 'valid_auc': 0.46633344821304945}
{'variant': 'pooled_mlp_rawfeat', 'epoch': 2, 'train_loss': 0.7367700338363647, 'valid_loss': 0.897965669631958, 'valid_auc': 0.4763181630252174}
{'variant': 'pooled_mlp_rawfeat', 'epoch': 3, 'train_loss': 0.6850089430809021, 'valid_loss': 0.8680045008659363, 'valid_auc': 0.48923531385340013}
{'variant': 'pooled_mlp_rawfeat', 'epoch': 4, 'train_loss': 0.6520435810089111, 'valid_loss': 0.8454210162162781, 'valid_auc': 0.5039375601182162}
{'variant': 'pooled_mlp_rawfeat', 'epoch': 5, 'train_loss': 0.6255509853363037, 'valid_loss': 0.8288470506668091, 'valid_auc': 0.5185246609965257}
{'variant': 'pooled_mlp_rawfeat', 'epoch': 6, 'train_loss': 0.6007488369941711, 'valid_loss': 0.8155401945114136, 'valid_auc': 0.5317851858828954}
{'variant': 'pooled_mlp_rawfeat', 'epoch': 7, 'train_loss': 0.5796998739242554, 'valid_loss': 0.8039997

In [9]:
def train_final_variant(variant: str) -> dict[str, tp.Any]:
    model, history = train_variant(
        variant=variant,
        emb_train=emb_files,
        raw_train=raw_files,
        labels_train=labels_files,
        site_train=site_ids_all,
        hour_train=hours_all,
        emb_valid=emb_files,
        raw_valid=raw_files,
        labels_valid=labels_files,
        site_valid=site_ids_all,
        hour_valid=hours_all,
    )
    payload = {
        'model_state_dict': model.state_dict(),
        'cfg': asdict(CFG),
        'variant': variant,
        'classes': PRIMARY_LABELS,
        'site_to_idx': site_to_idx,
    }
    out_path = OUTPUT_DIR / f'final_model_{variant}.pt'
    torch.save(payload, out_path)
    summary = {
        'final_variant': variant,
        'final_train_best_auc': float(max(history['valid_auc'])),
        'final_model_path': str(out_path),
    }
    save_json(summary, OUTPUT_DIR / 'final_model_summary.json')
    return summary


if RUN_FINAL_TRAIN:
    final_summary = train_final_variant(BEST_VARIANT_FOR_FINAL)
    print(json.dumps(final_summary, indent=2))
else:
    print('Final train skipped. Enable RUN_FINAL_TRAIN after OOF identifies a winning variant.')


Final train skipped. Enable RUN_FINAL_TRAIN after OOF identifies a winning variant.
